# P10.6-AI — Notebook 55: entrenamiento de estenosis central

Entrena el primer clasificador RSNA 2.5D sobre Sagittal T2/STIR. Usa train y validation del Notebook 54; el internal test queda reservado para el Notebook 56.

`humanReviewRequired=true` · `notClinicalDiagnosis=true`


In [1]:
# 1) Dependencias mínimas
from __future__ import annotations
import importlib.util, subprocess, sys
packages = {"pydicom":"pydicom","timm":"timm","tqdm":"tqdm","sklearn":"scikit-learn"}
missing = [pkg for mod,pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])


In [2]:
# 2) GPU y Drive
import getpass, json, os, subprocess
from pathlib import Path
import torch
from google.colab import drive  # type: ignore

if not torch.cuda.is_available():
    raise RuntimeError("Seleccioná GPU T4 o superior en el runtime de Colab.")
print({"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__})
drive.mount("/content/drive", force_remount=False)


{'gpu': 'Tesla T4', 'torch': '2.11.0+cu128'}
Mounted at /content/drive


In [3]:
# 3) Clonar/actualizar la rama e importar el pipeline
REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_REF = "enzo/p10-6-ai-rsna-findings"

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_ROOT)])
else:
    subprocess.check_call(["git","fetch","origin"], cwd=REPO_ROOT)
    subprocess.check_call(["git","checkout",REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git","pull","--ff-only"], cwd=REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_central_training import (
    TrainConfig, attach_coordinates, build_cache, ensure_local_subset,
    load_manifests, sha256_file, train,
)
print({"repoSha": subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_ROOT,text=True).strip()})


{'repoSha': '787474e3fa5e7011f26a3dc5aa1778c7ba8b5ae0'}


In [4]:
# 4) Rutas y configuración
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
SPLIT_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings" / "notebook54_split"
MODEL_ROOT = PFI_ROOT / "models" / "P10_6_rsna_findings" / "central_stenosis_sagittal_t2_2p5d"
CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"
RUN_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings" / "notebook55_training"
LOCAL_ROOT = Path("/content/RSNA_LUMBAR_DISC")
CACHE_ROOT = Path("/content/rsna_central_stenosis_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"
CFG = TrainConfig()

for path in (MODEL_ROOT, CHECKPOINT_ROOT, RUN_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print(CFG)


TrainConfig(seed=2026, image_size=224, crop_size=256, batch_size=32, num_workers=2, max_epochs=12, patience=4, learning_rate=0.0002, weight_decay=0.0001, model_name='efficientnet_b0', pretrained=True)


In [5]:
# 5) Cargar manifests aprobados sin leer internal_test
train_manifest, validation_manifest, split_summary = load_manifests(SPLIT_ROOT)
print({
    "trainRows": len(train_manifest),
    "validationRows": len(validation_manifest),
    "trainStudies": train_manifest.study_id.nunique(),
    "validationStudies": validation_manifest.study_id.nunique(),
    "internalTestAccessed": False,
})


{'trainRows': 6905, 'validationRows': 1480, 'trainStudies': 1381, 'validationStudies': 296, 'internalTestAccessed': False}


In [6]:
# 6) Preparar localmente solo las series requeridas
token = ""
if not (LOCAL_ROOT / "train_label_coordinates.csv").is_file():
    token = getpass.getpass("Pegá tu KAGGLE_API_TOKEN (no se mostrará): " ).strip()
    if not token:
        raise RuntimeError("No se ingresó token de Kaggle.")

ensure_local_subset(train_manifest, validation_manifest, LOCAL_ROOT, COMPETITION, token)
token = ""
print({"localRoot": str(LOCAL_ROOT), "ready": True})


Pegá tu KAGGLE_API_TOKEN (no se mostrará): ··········
Archivos extraídos: 5000
Archivos extraídos: 10000
Archivos extraídos: 15000
Archivos extraídos: 20000
Archivos extraídos: 25000
{'selectedFilesExtracted': 28598, 'minutes': 1.55}
{'localRoot': '/content/RSNA_LUMBAR_DISC', 'ready': True}


In [8]:
# 7) Unir coordenadas RSNA y construir cache 2.5D
#
# Algunas etiquetas de train.csv no tienen una coordenada puntual equivalente
# en train_label_coordinates.csv. Esas muestras se registran y se excluyen.
# No se inventan coordenadas ni se usa el centro de la imagen.

import pandas as pd


def attach_coordinates_safe(
    manifest: pd.DataFrame,
    local_root: Path,
    split_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    coordinates = pd.read_csv(
        local_root / "train_label_coordinates.csv",
        dtype={
            "study_id": str,
            "series_id": str,
        },
    )

    # Mantener únicamente coordenadas de estenosis central.
    coordinate_rows = coordinates.loc[
        coordinates["condition"]
        .astype(str)
        .str.strip()
        .eq("Spinal Canal Stenosis")
    ].copy()

    coordinate_rows["study_id"] = (
        coordinate_rows["study_id"].astype(str)
    )
    coordinate_rows["series_id"] = (
        coordinate_rows["series_id"].astype(str)
    )

    coordinate_rows["instance_number"] = pd.to_numeric(
        coordinate_rows["instance_number"],
        errors="coerce",
    )

    coordinate_rows["x"] = pd.to_numeric(
        coordinate_rows["x"],
        errors="coerce",
    )

    coordinate_rows["y"] = pd.to_numeric(
        coordinate_rows["y"],
        errors="coerce",
    )

    # Una sola coordenada por estudio, serie y nivel.
    coordinate_rows = (
        coordinate_rows
        .sort_values(
            [
                "study_id",
                "series_id",
                "level",
                "instance_number",
            ]
        )
        .drop_duplicates(
            [
                "study_id",
                "series_id",
                "level",
            ],
            keep="first",
        )
    )

    frame = manifest.copy()

    frame["study_id"] = frame["study_id"].astype(str)
    frame["series_id"] = frame["series_id"].astype(str)

    samples = frame.merge(
        coordinate_rows[
            [
                "study_id",
                "series_id",
                "level",
                "instance_number",
                "x",
                "y",
            ]
        ],
        on=[
            "study_id",
            "series_id",
            "level",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    missing_mask = (
        samples["instance_number"].isna()
        | samples["x"].isna()
        | samples["y"].isna()
    )

    missing_coordinates = samples.loc[
        missing_mask,
        [
            "study_id",
            "series_id",
            "level",
            "severity",
            "split",
        ],
    ].copy()

    missing_coordinates["reason"] = (
        "missing_rsna_label_coordinate"
    )

    missing_coordinates["excludedFromTraining"] = True

    usable_samples = samples.loc[
        ~missing_mask
    ].copy()

    usable_samples["instance_number"] = (
        usable_samples["instance_number"].astype(int)
    )

    usable_samples["severity_code"] = (
        usable_samples["severity"]
        .map(
            {
                "Normal/Mild": 0,
                "Moderate": 1,
                "Severe": 2,
            }
        )
    )

    if usable_samples["severity_code"].isna().any():
        unknown = (
            usable_samples.loc[
                usable_samples["severity_code"].isna(),
                "severity",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise RuntimeError(
            f"Severidades desconocidas: {unknown}"
        )

    usable_samples["severity_code"] = (
        usable_samples["severity_code"].astype(int)
    )

    usable_samples["local_series_path"] = (
        usable_samples.apply(
            lambda row: str(
                local_root
                / "train_images"
                / str(row["study_id"])
                / str(row["series_id"])
            ),
            axis=1,
        )
    )

    missing_series = usable_samples.loc[
        ~usable_samples["local_series_path"]
        .map(lambda value: Path(value).is_dir())
    ].copy()

    if not missing_series.empty:
        raise RuntimeError(
            "Faltan series locales para "
            f"{len(missing_series)} muestras."
        )

    usable_samples = (
        usable_samples
        .drop(columns=["_merge"])
        .reset_index(drop=True)
    )

    missing_coordinates = (
        missing_coordinates
        .reset_index(drop=True)
    )

    missing_rate = (
        len(missing_coordinates)
        / max(len(samples), 1)
    )

    print({
        "split": split_name,
        "manifestRows": int(len(samples)),
        "usableSamples": int(len(usable_samples)),
        "missingCoordinates": int(
            len(missing_coordinates)
        ),
        "missingCoordinateRate": round(
            missing_rate,
            4,
        ),
        "studiesWithMissingCoordinates": int(
            missing_coordinates["study_id"].nunique()
        ),
    })

    # Gate sanitario: bloquear si la pérdida fuera demasiado grande.
    if missing_rate > 0.02:
        raise RuntimeError(
            f"{split_name}: se excluyó "
            f"{missing_rate:.2%} por falta de coordenadas. "
            "Supera el límite permitido del 2 %."
        )

    return usable_samples, missing_coordinates


train_samples, train_missing_coordinates = (
    attach_coordinates_safe(
        train_manifest,
        LOCAL_ROOT,
        "train",
    )
)

validation_samples, validation_missing_coordinates = (
    attach_coordinates_safe(
        validation_manifest,
        LOCAL_ROOT,
        "validation",
    )
)

missing_coordinate_report = pd.concat(
    [
        train_missing_coordinates,
        validation_missing_coordinates,
    ],
    ignore_index=True,
)

missing_coordinate_report_path = (
    RUN_ROOT / "missing_coordinate_samples.csv"
)

missing_coordinate_report.to_csv(
    missing_coordinate_report_path,
    index=False,
)

print({
    "totalMissingCoordinates": int(
        len(missing_coordinate_report)
    ),
    "missingCoordinateReport": str(
        missing_coordinate_report_path
    ),
    "internalTestAccessed": False,
    "officialTestAccessed": False,
})

# Construcción de cache únicamente con muestras localizables.
build_cache(
    train_samples,
    CACHE_ROOT,
    "train",
    CFG,
)

build_cache(
    validation_samples,
    CACHE_ROOT,
    "validation",
    CFG,
)

print({
    "trainSamples": int(len(train_samples)),
    "validationSamples": int(
        len(validation_samples)
    ),
    "trainClasses": (
        train_samples["severity"]
        .value_counts()
        .to_dict()
    ),
    "validationClasses": (
        validation_samples["severity"]
        .value_counts()
        .to_dict()
    ),
})

{'split': 'train', 'manifestRows': 6905, 'usableSamples': 6815, 'missingCoordinates': 90, 'missingCoordinateRate': 0.013, 'studiesWithMissingCoordinates': 54}
{'split': 'validation', 'manifestRows': 1480, 'usableSamples': 1468, 'missingCoordinates': 12, 'missingCoordinateRate': 0.0081, 'studiesWithMissingCoordinates': 8}
{'totalMissingCoordinates': 102, 'missingCoordinateReport': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook55_training/missing_coordinate_samples.csv', 'internalTestAccessed': False, 'officialTestAccessed': False}


cache train:   0%|          | 0/6815 [00:00<?, ?it/s]

cache validation:   0%|          | 0/1468 [00:00<?, ?it/s]

{'trainSamples': 6815, 'validationSamples': 1468, 'trainClasses': {'Normal/Mild': 5969, 'Moderate': 517, 'Severe': 329}, 'validationClasses': {'Normal/Mild': 1287, 'Moderate': 108, 'Severe': 73}}


In [9]:
# 8) Entrenar y guardar checkpoints
manifest_hashes = {
    "train_manifest.csv": sha256_file(SPLIT_ROOT / "train_manifest.csv"),
    "validation_manifest.csv": sha256_file(SPLIT_ROOT / "validation_manifest.csv"),
    "split_summary.json": sha256_file(SPLIT_ROOT / "split_summary.json"),
}
result = train(
    train_samples,
    validation_samples,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
    RUN_ROOT,
    manifest_hashes,
    CFG,
)
print(json.dumps(result, indent=2))


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

  0%|          | 0/213 [00:00<?, ?it/s]

  0%|          | 0/46 [00:00<?, ?it/s]

{
  "epoch": 1,
  "train_macro_f1": 0.5359279588459259,
  "train_balanced_accuracy": 0.6287140855403067,
  "train_severe_recall": 0.7490601503759399,
  "train_loss": 1.2272198781054071,
  "validation_macro_f1": 0.35422330446464984,
  "validation_balanced_accuracy": 0.49530074187608436,
  "validation_severe_recall": 0.6164383561643836,
  "validation_loss": 1.3200113888657385,
  "selection_score": 0.46110330745685685
}


...

  0%|          | 0/213 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79b1845014e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^Exception ignored in: ^^
^<function _MultiProcessingDataLoaderIter.__del__ at 0x79b1845014e0>^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^    ^self._shutdown_workers()
^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only test a child process'if w.is_alive():

               ^ ^^^ ^ ^^^^^^^^^^^^^^^
^  F

  0%|          | 0/46 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79b1845014e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x79b1845014e0>
   Traceback (most recent call last):
   ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^    ^^self._shutdown_workers()
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():^
 ^ ^^ ^  
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^^assert self._parent_pid == os.getpid(), 'can only test a child process'
^ ^^  ^ ^^ ^ ^^ ^
   File "/usr/li

{
  "epoch": 7,
  "train_macro_f1": 0.9434761722827973,
  "train_balanced_accuracy": 0.9689665008286833,
  "train_severe_recall": 0.9932885906040269,
  "train_loss": 0.05435402335944809,
  "validation_macro_f1": 0.5399609483754072,
  "validation_balanced_accuracy": 0.5471010893385323,
  "validation_severe_recall": 0.3972602739726027,
  "validation_loss": 1.6312912938664654,
  "selection_score": 0.4985787742471908
}
{
  "bestEpoch": 3,
  "bestSelectionScore": 0.5509844733090432,
  "epochsCompleted": 7
}


In [10]:
# 9) Gate para Notebook 56
best_checkpoint = CHECKPOINT_ROOT / "best_checkpoint.pt"
history = RUN_ROOT / "training_history.csv"
checks = {
    "bestCheckpointExists": best_checkpoint.is_file(),
    "historyExists": history.is_file(),
    "studyLeakage": bool(set(train_samples.study_id) & set(validation_samples.study_id)),
    "internalTestAccessed": False,
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}
failures = [k for k,v in checks.items() if k not in {"studyLeakage","internalTestAccessed","officialTestAccessed"} and v is not True]
failures += [k for k in ("studyLeakage","internalTestAccessed","officialTestAccessed") if checks[k] is not False]
print(json.dumps(checks, indent=2))
if failures:
    raise RuntimeError("Notebook 56 no habilitado: " + ", ".join(failures))
print({
    "status": "APPROVED_FOR_NOTEBOOK_56",
    "bestCheckpoint": str(best_checkpoint),
    "plannedFinalArtifact": "rsna_central_stenosis_sagittal_t2_2p5d.pt",
})


{
  "bestCheckpointExists": true,
  "historyExists": true,
  "studyLeakage": false,
  "internalTestAccessed": false,
  "officialTestAccessed": false,
  "humanReviewRequired": true,
  "notClinicalDiagnosis": true
}
{'status': 'APPROVED_FOR_NOTEBOOK_56', 'bestCheckpoint': '/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings/central_stenosis_sagittal_t2_2p5d/checkpoints/best_checkpoint.pt', 'plannedFinalArtifact': 'rsna_central_stenosis_sagittal_t2_2p5d.pt'}
